In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("DistributedAnalytics_Member3") \
    .getOrCreate()

In [30]:
# Data Loading
df = spark.read.parquet("work/data/clean_reviews.parquet")
df_business = spark.read.parquet("work/data/clean_business.parquet")

# Show the schema if it successfully reads
df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- clean_text: string (nullable = true)



In [ ]:
#large-scale aggregation & traffic statistics

print("Large-Scale Aggregation & Traffic Statistics...")
start_time = time.time()


#Calculate the total number of review actions globally across the cluster
total_reviews_count = df.count()



print(f"\n--- Global Traffic Metrics ---")
print(f"Total processed log actions: {total_reviews_count:,} rows.")



# Basic rating distribution statistics
rating_distribution = df.groupBy("stars").count().orderBy(F.desc("stars"))
rating_distribution.show()



# Export data distribution to the submit folder
rating_distribution.coalesce(1).write.mode("overwrite") \
    .option("header", "true").csv("work/submit/traffic_statistics_report")


end_time = time.time()
print(f"⏱️ Performance Metric: Completed in {end_time - start_time:.2f} seconds.")

Large-Scale Aggregation & Traffic Statistics...

--- Global Traffic Metrics ---
Total processed log actions: 2,798,412 rows.
+-----+-------+
|stars|  count|
+-----+-------+
|  5.0|1293294|
|  4.0| 582617|
|  3.0| 276599|
|  2.0| 217693|
|  1.0| 428209|
+-----+-------+

⏱️ Performance Metric: Completed in 5.28 seconds.


In [ ]:
#hourly trends

print(" Hourly Trends")
start_time = time.time()

# Extract the hour component from the date timestamp metadata
df_with_hour = df.withColumn("hour_of_day", F.hour("date"))

# Group by hour and count the total review interaction records
hourly_traffic = df_with_hour.groupBy("hour_of_day") \
    .count() \
    .withColumnRenamed("count", "total_reviews") \
    .orderBy("hour_of_day")




#  Output results directly to the console interface
hourly_traffic.show(24)

# Programmatically isolate the highest operational traffic spike
peak_hour_row = hourly_traffic.orderBy(F.desc("total_reviews")).first()
print(f"--- Temporal Traffic Insights ---")
print(f"Peak Traffic Window: Hour {peak_hour_row['hour_of_day']}:00")
print(f"Peak Record Volume: {peak_hour_row['total_reviews']:,} actions.")


# Export the clean matrix to the host file system folder
hourly_traffic.coalesce(1).write.mode("overwrite") \
    .option("header", "true").csv("work/submit/hourly_trends_report")

end_time = time.time()
print(f"⏱️ Performance Metric: Completed in {end_time - start_time:.2f} seconds.")

 Hourly Trends
+-----------+-------------+
|hour_of_day|total_reviews|
+-----------+-------------+
|          0|       184279|
|          1|       182832|
|          2|       166837|
|          3|       136134|
|          4|        99246|
|          5|        65054|
|          6|        40301|
|          7|        24003|
|          8|        15314|
|          9|        12156|
|         10|        15244|
|         11|        28891|
|         12|        54131|
|         13|        89406|
|         14|       119535|
|         15|       143414|
|         16|       162124|
|         17|       176917|
|         18|       186343|
|         19|       186028|
|         20|       181064|
|         21|       175513|
|         22|       173887|
|         23|       179759|
+-----------+-------------+

--- Temporal Traffic Insights ---
Peak Traffic Window: Hour 18:00
Peak Record Volume: 186,343 actions.
⏱️ Performance Metric: Completed in 5.71 seconds.


In [ ]:
#city analysis

print(" City Analysis...")
start_time = time.time()

#Execute a distributed relational inner join across reviews and business metadata
df_joined = df.join(df_business, on="business_id", how="inner")

#Group review footprints by business city location values
city_analysis = df_joined.groupBy("city") \
    .count() \
    .withColumnRenamed("count", "review_count") \
    .orderBy(F.desc("review_count"))

# Present the top 10 most active city regions
city_analysis.show(10)

# Export regional metrics back to your project workspace
city_analysis.coalesce(1).write.mode("overwrite") \
    .option("header", "true").csv("work/submit/city_analysis_report")

end_time = time.time()
print(f"⏱️ Performance Metric: Completed in {end_time - start_time:.2f} seconds.")

 City Analysis...
+-------------+------------+
|         city|review_count|
+-------------+------------+
| Philadelphia|      387623|
|  New Orleans|      254597|
|        Tampa|      182367|
|    Nashville|      180748|
|       Tucson|      162511|
| Indianapolis|      144625|
|         Reno|      140650|
|Santa Barbara|      107613|
|  Saint Louis|      101399|
|        Boise|       42042|
+-------------+------------+
only showing top 10 rows

⏱️ Performance Metric: Completed in 6.75 seconds.


In [ ]:
# provider comparisons

print("Provider Comparisons...")
start_time = time.time()


df_exploded_categories = df_joined.filter(df_joined["categories"].isNotNull()) \
    .withColumn("individual_category", F.explode(F.split(F.col("categories"), ",\s*")))

#Calculate engagement weights by sorting top provider domains
provider_comparisons = df_exploded_categories.groupBy("individual_category") \
    .count() \
    .withColumnRenamed("count", "review_count") \
    .orderBy(F.desc("review_count"))


# Output the top 15 categorical provider channels
provider_comparisons.show(15, truncate=False)

# Export industry metric data out to the submission folder
provider_comparisons.coalesce(1).write.mode("overwrite") \
    .option("header", "true").csv("work/submit/provider_comparisons_report")



end_time = time.time()
print(f"⏱️ Performance Metric: Completed in {end_time - start_time:.2f} seconds.")

Provider Comparisons...
+-------------------------+------------+
|individual_category      |review_count|
+-------------------------+------------+
|Restaurants              |1891134     |
|Food                     |725319      |
|Nightlife                |616684      |
|Bars                     |582921      |
|American (Traditional)   |405158      |
|American (New)           |393803      |
|Breakfast & Brunch       |347120      |
|Sandwiches               |277131      |
|Seafood                  |248402      |
|Event Planning & Services|244658      |
|Shopping                 |209512      |
|Pizza                    |190630      |
|Burgers                  |178376      |
|Coffee & Tea             |176338      |
|Italian                  |176208      |
+-------------------------+------------+
only showing top 15 rows

⏱️ Performance Metric: Completed in 9.24 seconds.
